In [1]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/08 20:59:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
df_green = spark.read \
    .option("recursiveFileLookup", "true") \
    .parquet('data/pq/green/')

In [5]:
df_green.createOrReplaceTempView('green')

In [6]:
df_green.columns

['VendorID',
 'lpep_pickup_datetime',
 'lpep_dropoff_datetime',
 'store_and_fwd_flag',
 'RatecodeID',
 'PULocationID',
 'DOLocationID',
 'passenger_count',
 'trip_distance',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'ehail_fee',
 'improvement_surcharge',
 'total_amount',
 'payment_type',
 'trip_type',
 'congestion_surcharge',
 'cbd_congestion_fee']

In [7]:
df_green_revenue = spark.sql("""
SELECT
    -- Revenue grouping
    date_trunc('hour', lpep_pickup_datetime) AS hour,
    PULocationID AS zone,
    
    -- Revenue calculation
    SUM(total_amount) AS amount,
    COUNT(*) AS number_records
FROM
    green
WHERE
    lpep_pickup_datetime >= '2025-11-01 00:00:00'
GROUP BY
    PULocationID, date_trunc('hour', lpep_pickup_datetime)
ORDER BY hour, zone;
""")

In [8]:
df_green_revenue.show()

+-------------------+----+------------------+--------------+
|               hour|zone|            amount|number_records|
+-------------------+----+------------------+--------------+
|2025-11-01 00:00:00|   7|51.480000000000004|             2|
|2025-11-01 00:00:00|  20|            189.53|             1|
|2025-11-01 00:00:00|  25|             70.27|             3|
|2025-11-01 00:00:00|  37|             19.44|             1|
|2025-11-01 00:00:00|  41| 81.02000000000001|             3|
|2025-11-01 00:00:00|  49|             32.99|             1|
|2025-11-01 00:00:00|  50|              16.0|             1|
|2025-11-01 00:00:00|  61|             75.75|             2|
|2025-11-01 00:00:00|  65|              93.9|             3|
|2025-11-01 00:00:00|  66|             65.28|             2|
|2025-11-01 00:00:00|  74| 99.30999999999999|             7|
|2025-11-01 00:00:00|  75|119.41000000000001|             5|
|2025-11-01 00:00:00|  82|            152.82|             3|
|2025-11-01 00:00:00|  8

In [16]:
df_green_revenue \
    .repartition(20) \
    .write.parquet('data/report/revenue/green/', mode='overwrite')

In [11]:
df_yellow = spark.read \
    .option("recursiveFileLookup", "true") \
    .parquet('data/pq/yellow/')
df_yellow.createOrReplaceTempView('yellow')

In [12]:
df_yellow.createOrReplaceTempView('yellow')

In [13]:
df_yellow_revenue = spark.sql("""
SELECT
    -- Revenue grouping
    date_trunc('hour', tpep_pickup_datetime) AS hour,
    PULocationID AS zone,
    
    -- Revenue calculation
    SUM(total_amount) AS amount,
    COUNT(*) AS number_records
FROM
    yellow
WHERE
    tpep_pickup_datetime >= '2025-11-01 00:00:00'
GROUP BY
    PULocationID, date_trunc('hour', tpep_pickup_datetime)
ORDER BY hour, zone;
""")

In [15]:
df_yellow_revenue \
    .repartition(20) \
    .write.parquet('data/report/revenue/yellow/', mode='overwrite')

In [20]:
df_green_revenue_tmp = df_green_revenue \
    .withColumnRenamed('amount', 'green_amount') \
    .withColumnRenamed('number_records', 'green_number_records')

df_yellow_revenue_tmp = df_yellow_revenue \
    .withColumnRenamed('amount', 'yellow_amount') \
    .withColumnRenamed('number_records', 'yellow_number_records')

In [21]:
df_join = df_green_revenue_tmp.join(df_yellow_revenue_tmp, on=['hour', 'zone'], how='outer')

In [22]:
df_join.show()

[Stage 55:=============================>                            (1 + 1) / 2]

+-------------------+----+------------+--------------------+-------------+---------------------+
|               hour|zone|green_amount|green_number_records|yellow_amount|yellow_number_records|
+-------------------+----+------------+--------------------+-------------+---------------------+
|2025-11-01 00:00:00|   1|        NULL|                NULL|       197.11|                    1|
|2025-11-01 03:00:00|   1|        NULL|                NULL|       143.39|                    1|
|2025-11-01 07:00:00|   1|        NULL|                NULL|        175.2|                    1|
|2025-11-01 11:00:00|   1|        NULL|                NULL|       175.27|                    1|
|2025-11-01 13:00:00|   1|        NULL|                NULL|        146.4|                    1|
|2025-11-01 19:00:00|   1|        NULL|                NULL|       513.94|                    4|
|2025-11-01 20:00:00|   1|        NULL|                NULL|        139.2|                    1|
|2025-11-01 21:00:00|   1|    

In [26]:
df_join.write.parquet('data/report/revenue/total', mode='overwrite')

In [27]:
df_green_revenue = spark.read.parquet('data/report/revenue/green')
df_yellow_revenue = spark.read.parquet('data/report/revenue/yellow')

In [28]:
df_green_revenue_tmp = df_green_revenue \
    .withColumnRenamed('amount', 'green_amount') \
    .withColumnRenamed('number_records', 'green_number_records')

df_yellow_revenue_tmp = df_yellow_revenue \
    .withColumnRenamed('amount', 'yellow_amount') \
    .withColumnRenamed('number_records', 'yellow_number_records')

In [29]:
df_join = df_green_revenue_tmp.join(df_yellow_revenue_tmp, on=['hour', 'zone'], how='outer')

In [30]:
df_join.write.parquet('data/report/revenue/total', mode='overwrite')

In [31]:
df_join = spark.read.parquet('data/report/revenue/total')

In [32]:
df_join.show()

+-------------------+----+------------------+--------------------+-------------------+---------------------+
|               hour|zone|      green_amount|green_number_records|      yellow_amount|yellow_number_records|
+-------------------+----+------------------+--------------------+-------------------+---------------------+
|2025-11-01 00:00:00|   7|51.480000000000004|                   2|             342.73|                   38|
|2025-11-01 00:00:00|  11|              NULL|                NULL|               9.75|                    1|
|2025-11-01 00:00:00|  13|              NULL|                NULL|              471.6|                   20|
|2025-11-01 00:00:00|  18|              NULL|                NULL|               6.23|                    2|
|2025-11-01 00:00:00|  21|              NULL|                NULL|               5.25|                    2|
|2025-11-01 00:00:00|  23|              NULL|                NULL|               2.27|                    1|
|2025-11-01 00:00:0

In [34]:
df_zones = spark.read.parquet('zones/')

In [35]:
df_zones.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [36]:
df_result = df_join.join(df_zones, df_join.zone == df_zones.LocationID)

In [41]:
df_result.drop('zone').show()

+-------------------+------------------+--------------------+-------------------+---------------------+----------+-------------+------------+
|               hour|      green_amount|green_number_records|      yellow_amount|yellow_number_records|LocationID|      Borough|service_zone|
+-------------------+------------------+--------------------+-------------------+---------------------+----------+-------------+------------+
|2025-11-01 00:00:00|51.480000000000004|                   2|             342.73|                   38|         7|       Queens|   Boro Zone|
|2025-11-01 00:00:00|              NULL|                NULL|               9.75|                    1|        11|     Brooklyn|   Boro Zone|
|2025-11-01 00:00:00|              NULL|                NULL|              471.6|                   20|        13|    Manhattan| Yellow Zone|
|2025-11-01 00:00:00|              NULL|                NULL|               6.23|                    2|        18|        Bronx|   Boro Zone|
|2025-

In [40]:
df_result.drop('zone').write.parquet('tmp/revenue-zones')